# 💄 뷰티 브랜드 UGC 영상 콘텐츠 생성 AI

제품 이미지 + 기획초안 → **30초 UGC 쇼츠 영상** 자동 생성 파이프라인

## 전체 파이프라인
```
[입력] 제품 이미지 + 기획초안
    ↓
  [A] Claude: UGC 콘텐츠 기획안 생성 (4~8초 장면 × 5~6개 = 30초)
    ↓
  [B] DALL-E 3: 가상 크리에이터 모델 이미지 생성
    ↓
  [C] DALL-E 3: 각 장면별 1st Scene / Last Scene 이미지 생성
    ↓
  [D] Google Veo3: 장면별 영상 생성 (1st+Last scene 이미지 + 기획안 사용)
    ↓
  [E] moviepy: 장면 영상 결합 → 최종 30초 영상
```

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJaeWon/Astar/blob/claude/beauty-ugc-video-generator-i10mH/beauty_ugc_video_generator.ipynb)

## 0. 의존성 설치

In [ ]:
!pip install -q anthropic openai google-genai moviepy Pillow requests python-dotenv

## 1. 설정 및 API 키 입력

In [ ]:
import os
import json
import base64
import time
import re
import requests
import textwrap
from pathlib import Path
from PIL import Image
from io import BytesIO
from IPython.display import display, Image as IPImage, Video, Markdown

import anthropic
import openai
from google import genai
from google.genai import types
import moviepy.editor as mpe

# ── API 키 설정 ──────────────────────────────────────────────
# Google Colab 환경이라면 userdata를 사용하세요:
# from google.colab import userdata
# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
# OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')
# GOOGLE_API_KEY    = userdata.get('GOOGLE_API_KEY')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', 'YOUR_ANTHROPIC_API_KEY')
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY',    'YOUR_OPENAI_API_KEY')
GOOGLE_API_KEY    = os.getenv('GOOGLE_API_KEY',    'YOUR_GOOGLE_API_KEY')

# ── 클라이언트 초기화 ─────────────────────────────────────────
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
veo_client    = genai.Client(api_key=GOOGLE_API_KEY)

# ── 출력 디렉토리 ─────────────────────────────────────────────
OUTPUT_DIR = Path('ugc_output')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'images').mkdir(exist_ok=True)
(OUTPUT_DIR / 'videos').mkdir(exist_ok=True)

print('✅ 초기화 완료')

## 2. 입력값 설정 — 제품 이미지 + 기획초안

In [ ]:
# ── [선택 1] URL로 제품 이미지 지정 ───────────────────────────
PRODUCT_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/44/SK-II_Facial_Treatment_Essence.jpg/440px-SK-II_Facial_Treatment_Essence.jpg"

# ── [선택 2] 로컬 파일 경로 (URL 대신 사용 가능) ──────────────
# PRODUCT_IMAGE_PATH = "/content/product.jpg"

# ── 사용자 기획초안 입력 ──────────────────────────────────────
USER_BRIEF = """
브랜드: 글로우랩 (GlowLab)
제품: 비타민C 브라이트닝 세럼 30ml
타겟: 20~30대 여성, 칙칙하고 피로해 보이는 피부 고민
핵심 메시지: 단 하루만에 달라지는 피부 광채
톤&무드: 밝고 생동감 있는, 진정성 있는 일상 룩
CTA: 링크 클릭 → 첫 구매 20% 할인
"""

# ── 제품 이미지 로드 유틸 ─────────────────────────────────────
def load_image_as_base64(url: str = None, path: str = None) -> tuple[str, str]:
    """이미지를 base64로 인코딩하고 (data, media_type) 반환"""
    if path:
        with open(path, 'rb') as f:
            data = f.read()
        ext = Path(path).suffix.lower().lstrip('.')
        media_type = f'image/{ext if ext != "jpg" else "jpeg"}'
    else:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        data = resp.content
        ct = resp.headers.get('content-type', 'image/jpeg')
        media_type = ct.split(';')[0].strip()
    return base64.b64encode(data).decode(), media_type

product_b64, product_mime = load_image_as_base64(url=PRODUCT_IMAGE_URL)
print(f'✅ 제품 이미지 로드 완료 (mime: {product_mime}, size: {len(product_b64)//1024}KB)')
display(IPImage(data=base64.b64decode(product_b64), width=300))

## STEP A — UGC 콘텐츠 기획안 생성 (Claude)
제품 이미지 + 기획초안 → **4~8초 장면 5~6개** (총 30초 내외) 구성

In [ ]:
PLAN_SYSTEM_PROMPT = """\
당신은 뷰티 브랜드 전문 UGC(User Generated Content) 콘텐츠 기획자입니다.
제품 이미지와 기획초안을 분석해 틱톡/인스타그램 릴스용 30초 UGC 쇼츠 기획안을 작성하세요.

출력 형식: 반드시 아래 JSON 배열만 출력하세요. 다른 텍스트는 포함하지 마세요.
[
  {
    "scene_number": 1,
    "duration_sec": 5,
    "title": "장면 제목",
    "description": "장면 내용 및 행동 묘사 (한국어)",
    "voiceover": "내레이션/자막 텍스트 (한국어)",
    "visual_style": "시각적 스타일 및 카메라 무드 (한국어)",
    "product_focus": "제품 노출 방식 (한국어)",
    "first_scene_prompt": "DALL-E용 1st frame 이미지 프롬프트 (영어, 상세)",
    "last_scene_prompt": "DALL-E용 last frame 이미지 프롬프트 (영어, 상세)"
  }
]

규칙:
- 장면 수: 5~6개, 각 장면 4~8초, 총합 28~32초
- UGC 특유의 진정성 있는 스타일 (과도한 광고 느낌 지양)
- first/last_scene_prompt는 DALL-E 3에 바로 입력할 수 있는 영어 프롬프트
- 크리에이터는 실제 인물처럼 자연스럽게 등장
"""

def generate_ugc_plan(brief: str, image_b64: str, image_mime: str) -> list[dict]:
    """STEP A: Claude로 UGC 기획안 생성"""
    response = claude_client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=4096,
        system=PLAN_SYSTEM_PROMPT,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': image_mime,
                            'data': image_b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': f'아래 기획초안과 위 제품 이미지를 바탕으로 UGC 기획안을 JSON으로 작성해주세요.\n\n기획초안:\n{brief}'
                    }
                ]
            }
        ]
    )
    raw = response.content[0].text.strip()
    # JSON 블록 추출
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if match:
        raw = match.group(0)
    return json.loads(raw)

print('⏳ STEP A: UGC 기획안 생성 중...')
scenes = generate_ugc_plan(USER_BRIEF, product_b64, product_mime)

# 결과 저장 및 출력
with open(OUTPUT_DIR / 'ugc_plan.json', 'w', encoding='utf-8') as f:
    json.dump(scenes, f, ensure_ascii=False, indent=2)

total_sec = sum(s['duration_sec'] for s in scenes)
print(f'\n✅ STEP A 완료: {len(scenes)}개 장면, 총 {total_sec}초\n')
for s in scenes:
    print(f"  [{s['scene_number']}] {s['title']} ({s['duration_sec']}초)")
    print(f"      내용: {s['description'][:60]}...")
    print(f"      자막: {s['voiceover'][:50]}...\n")

## STEP B — 가상 크리에이터 모델 이미지 생성 (DALL-E 3)
브랜드 타겟에 맞는 **일관성 있는 UGC 크리에이터 페르소나** 이미지 생성

In [ ]:
CREATOR_PERSONA_PROMPT = """\
A natural, authentic-looking Korean female beauty content creator in her mid-to-late 20s.
She has a bright, fresh complexion with minimal makeup — dewy skin, soft pink lips, light mascara.
Her hair is dark brown, slightly wavy, shoulder length, casually styled.
She is wearing a cozy white oversized knit sweater.
She is smiling warmly at the camera in a bright, minimal Korean apartment setting with soft morning light.
Vertical 9:16 frame, shot on iPhone, realistic photo style, high quality, UGC aesthetic.
The image should feel like an authentic selfie taken by the creator herself.
"""

def generate_creator_image(prompt: str, filename: str) -> tuple[str, Image.Image]:
    """STEP B: DALL-E 3으로 크리에이터 이미지 생성. (path, PIL Image) 반환"""
    response = openai_client.images.generate(
        model='dall-e-3',
        prompt=prompt,
        size='1024x1792',   # 9:16 세로형
        quality='hd',
        style='natural',
        n=1
    )
    img_url = response.data[0].url
    img_data = requests.get(img_url, timeout=30).content
    path = OUTPUT_DIR / 'images' / filename
    with open(path, 'wb') as f:
        f.write(img_data)
    return str(path), Image.open(BytesIO(img_data))

print('⏳ STEP B: 가상 크리에이터 모델 이미지 생성 중...')
creator_path, creator_img = generate_creator_image(CREATOR_PERSONA_PROMPT, 'creator_model.png')
print(f'✅ STEP B 완료: {creator_path}')
display(creator_img.resize((250, 444)))

## STEP C — 장면별 1st / Last Scene 이미지 생성 (DALL-E 3)
A 기획안 + B 크리에이터 이미지를 기반으로 각 장면의 **시작·끝 프레임** 생성

In [ ]:
CREATOR_STYLE_SUFFIX = """
The creator is the same Korean female in her late 20s with dewy skin, dark wavy shoulder-length hair,
minimal makeup, wearing a white oversized knit sweater.
Vertical 9:16, realistic photo, high quality, authentic UGC aesthetic, shot on iPhone.
"""

def generate_scene_frame(
    scene: dict,
    frame_type: str  # 'first' or 'last'
) -> tuple[str, Image.Image]:
    """장면 하나의 first 또는 last frame 이미지 생성"""
    base_prompt = (
        scene['first_scene_prompt']
        if frame_type == 'first'
        else scene['last_scene_prompt']
    )
    full_prompt = base_prompt + CREATOR_STYLE_SUFFIX

    response = openai_client.images.generate(
        model='dall-e-3',
        prompt=full_prompt[:3900],   # DALL-E 4000자 제한
        size='1024x1792',
        quality='hd',
        style='natural',
        n=1
    )
    img_url = response.data[0].url
    img_data = requests.get(img_url, timeout=30).content
    fname = f'scene_{scene["scene_number"]:02d}_{frame_type}.png'
    path = OUTPUT_DIR / 'images' / fname
    with open(path, 'wb') as f:
        f.write(img_data)
    return str(path), Image.open(BytesIO(img_data))

print('⏳ STEP C: 각 장면 이미지 생성 중 (장면당 ~15초)...')
scene_images = []  # [{'scene': ..., 'first_path': ..., 'last_path': ...}]

for scene in scenes:
    n = scene['scene_number']
    print(f'  Scene {n}/{len(scenes)}: {scene["title"]}')

    first_path, first_img = generate_scene_frame(scene, 'first')
    print(f'    ✓ 1st frame 저장: {first_path}')
    time.sleep(1)  # Rate limit 방지

    last_path, last_img = generate_scene_frame(scene, 'last')
    print(f'    ✓ Last frame 저장: {last_path}')
    time.sleep(1)

    scene_images.append({
        'scene': scene,
        'first_path': first_path,
        'last_path': last_path,
        'first_img': first_img,
        'last_img': last_img
    })

print(f'\n✅ STEP C 완료: {len(scene_images)}개 장면 × 2프레임 = {len(scene_images)*2}장 이미지 생성')

# 생성된 이미지 미리보기
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(scene_images), 2, figsize=(8, len(scene_images) * 5))
for i, si in enumerate(scene_images):
    axes[i][0].imshow(si['first_img'].resize((360, 640)))
    axes[i][0].set_title(f'Scene {i+1} | 1st Frame', fontsize=9)
    axes[i][0].axis('off')
    axes[i][1].imshow(si['last_img'].resize((360, 640)))
    axes[i][1].set_title(f'Scene {i+1} | Last Frame', fontsize=9)
    axes[i][1].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'images' / 'scene_frames_preview.png', dpi=80)
plt.show()

## STEP D — 장면별 영상 생성 (Google Veo3)
C의 1st Scene + Last Scene 이미지 + A의 기획안 → 각 장면 영상 생성

In [ ]:
def image_path_to_part(image_path: str) -> types.Part:
    """이미지 파일을 Veo3 API용 types.Part로 변환"""
    with open(image_path, 'rb') as f:
        img_bytes = f.read()
    return types.Part.from_bytes(data=img_bytes, mime_type='image/png')

def build_veo_prompt(scene: dict) -> str:
    """기획안 장면 정보를 Veo3 프롬프트로 변환"""
    return (
        f"{scene['description']} "
        f"Visual style: {scene['visual_style']}. "
        f"Product focus: {scene['product_focus']}. "
        f"Authentic UGC beauty content, vertical 9:16, shot on iPhone, "
        f"natural lighting, realistic, high quality short-form video."
    )

def generate_scene_video(
    scene: dict,
    first_image_path: str,
    last_image_path: str,
    output_path: str,
    poll_interval: int = 10,
    max_wait: int = 600
) -> str:
    """Veo3으로 단일 장면 영상 생성 후 파일 저장. 저장 경로 반환."""
    prompt_text = build_veo_prompt(scene)
    duration = min(max(scene['duration_sec'], 4), 8)  # 4~8초 클램프

    # 시작/끝 프레임 이미지를 Parts로 변환
    first_part = image_path_to_part(first_image_path)
    last_part  = image_path_to_part(last_image_path)

    operation = veo_client.models.generate_videos(
        model='veo-3.0-generate-preview',
        prompt=prompt_text,
        image=first_part,          # 시작 프레임
        config=types.GenerateVideosConfig(
            number_of_videos=1,
            duration_seconds=duration,
            aspect_ratio='9:16',
            person_generation='allow_adult',
            end_image=last_part,   # 끝 프레임
        )
    )

    # 비동기 작업 폴링
    waited = 0
    while not operation.done:
        if waited >= max_wait:
            raise TimeoutError(f'Scene {scene["scene_number"]} 영상 생성 타임아웃 ({max_wait}초)')
        print(f'    ⏳ 생성 중... ({waited}s)', end='\r')
        time.sleep(poll_interval)
        operation = veo_client.operations.get(operation)
        waited += poll_interval

    # 결과 저장
    video = operation.response.generated_videos[0]
    veo_client.files.download(file=video.video)
    video.video.save(output_path)
    return output_path

print('⏳ STEP D: 장면별 Veo3 영상 생성 중...')
video_paths = []

for si in scene_images:
    scene = si['scene']
    n = scene['scene_number']
    out_path = str(OUTPUT_DIR / 'videos' / f'scene_{n:02d}.mp4')
    print(f'\n  [Scene {n}] {scene["title"]} ({scene["duration_sec"]}초)')
    try:
        path = generate_scene_video(
            scene=scene,
            first_image_path=si['first_path'],
            last_image_path=si['last_path'],
            output_path=out_path
        )
        video_paths.append(path)
        print(f'    ✅ 저장 완료: {path}')
    except Exception as e:
        print(f'    ❌ 오류: {e}')

print(f'\n✅ STEP D 완료: {len(video_paths)}/{len(scene_images)}개 장면 영상 생성')

## STEP E — 장면 영상 결합 → 최종 30초 UGC 영상 (moviepy)
생성된 장면 영상들을 하나의 영상으로 합칩니다.

In [ ]:
def combine_videos(
    video_paths: list[str],
    output_path: str,
    transition_duration: float = 0.3,
    target_resolution: tuple = (1080, 1920)  # 9:16 FHD
) -> str:
    """STEP E: 장면 영상 결합. 간단한 크로스페이드 트랜지션 적용."""
    if not video_paths:
        raise ValueError('결합할 영상이 없습니다.')

    clips = []
    for i, vp in enumerate(video_paths):
        clip = mpe.VideoFileClip(vp)
        # 해상도 표준화 (모두 동일한 해상도로)
        w, h = target_resolution
        clip = clip.resize((w, h))
        # 트랜지션: 첫 장면 제외 페이드인, 마지막 장면 제외 페이드아웃
        if i > 0:
            clip = clip.fadein(transition_duration)
        if i < len(video_paths) - 1:
            clip = clip.fadeout(transition_duration)
        clips.append(clip)
        print(f'  ✓ Scene {i+1} 로드: {clip.duration:.1f}초')

    final = mpe.concatenate_videoclips(clips, method='compose')
    print(f'\n  결합 영상 총 길이: {final.duration:.1f}초')

    final.write_videofile(
        output_path,
        fps=30,
        codec='libx264',
        audio_codec='aac',
        preset='fast',
        verbose=False,
        logger=None
    )
    for c in clips:
        c.close()
    final.close()
    return output_path

FINAL_VIDEO_PATH = str(OUTPUT_DIR / 'final_ugc_video.mp4')

print('⏳ STEP E: 장면 영상 결합 중...')
if video_paths:
    final_path = combine_videos(video_paths, FINAL_VIDEO_PATH)
    print(f'\n✅ STEP E 완료! 최종 영상 저장: {final_path}')
    display(Video(final_path, width=300))
else:
    print('❌ 결합할 영상이 없습니다. STEP D를 먼저 완료해주세요.')

## 최종 결과 요약

In [ ]:
def print_summary(scenes: list[dict], video_paths: list[str], final_path: str):
    total_sec = sum(s['duration_sec'] for s in scenes)
    lines = [
        '## 🎬 UGC 영상 생성 완료',
        '',
        f'- 총 장면 수: **{len(scenes)}개**',
        f'- 총 길이: **{total_sec}초**',
        f'- 생성된 장면 영상: **{len(video_paths)}개**',
        f'- 최종 영상: `{final_path}`',
        '',
        '### 장면 구성',
    ]
    for s in scenes:
        lines.append(
            f"  - **Scene {s['scene_number']}** [{s['duration_sec']}초] "
            f"{s['title']} — {s['voiceover'][:40]}..."
        )
    lines += [
        '',
        '### 출력 파일',
        f'  - 기획안: `ugc_output/ugc_plan.json`',
        f'  - 크리에이터 이미지: `ugc_output/images/creator_model.png`',
        f'  - 장면 이미지: `ugc_output/images/scene_XX_first/last.png`',
        f'  - 장면 영상: `ugc_output/videos/scene_XX.mp4`',
        f'  - 최종 영상: `ugc_output/final_ugc_video.mp4`',
    ]
    display(Markdown('\n'.join(lines)))

print_summary(scenes, video_paths, FINAL_VIDEO_PATH)

---
## 부록: 부분 재생성 유틸리티
특정 장면만 다시 생성하고 싶을 때 사용

In [ ]:
def regenerate_scene(scene_number: int, regenerate_images: bool = True, regenerate_video: bool = True):
    """특정 장면 번호만 재생성"""
    idx = scene_number - 1
    if idx < 0 or idx >= len(scenes):
        print(f'❌ 유효하지 않은 장면 번호: {scene_number}')
        return

    scene = scenes[idx]
    si = scene_images[idx]
    print(f'🔄 Scene {scene_number} 재생성: {scene["title"]}')

    if regenerate_images:
        print('  이미지 재생성 중...')
        fp, fi = generate_scene_frame(scene, 'first')
        lp, li = generate_scene_frame(scene, 'last')
        si['first_path'], si['first_img'] = fp, fi
        si['last_path'],  si['last_img']  = lp, li
        print(f'  ✅ 이미지 저장: {fp}, {lp}')
        display(fi.resize((200, 356)))
        display(li.resize((200, 356)))

    if regenerate_video:
        print('  영상 재생성 중...')
        out_path = str(OUTPUT_DIR / 'videos' / f'scene_{scene_number:02d}.mp4')
        path = generate_scene_video(
            scene=scene,
            first_image_path=si['first_path'],
            last_image_path=si['last_path'],
            output_path=out_path
        )
        video_paths[idx] = path
        print(f'  ✅ 영상 저장: {path}')
        display(Video(path, width=200))

# 사용 예시: Scene 3만 재생성
# regenerate_scene(scene_number=3, regenerate_images=True, regenerate_video=True)